# AgriShift — 03. Feature Engineering

**Project:** Agricultural Diversification & Crop Pattern Intelligence in India  
**SDG:** 2 — Zero Hunger

## Purpose

This notebook converts the cleaned crop records into district-year agricultural portfolio features.

We will measure:
- How many different crops are recorded in each district-year.
- Each crop's share of the district-year's reported crop area.
- Whether crop area is concentrated in a few crops or more evenly distributed.
- How reported crop area is distributed across seasons.

## Important definition

We aggregate `Area` across seasons for the same crop before calculating annual crop shares. The resulting percentages describe each crop's share of the **total reported crop area across recorded seasons**. They do not describe the share of a district's unique physical land. The same land may be cultivated in more than one season.

We use `Area` for diversification features. `Production` is retained in the source data but is not used to calculate crop shares.

## Outputs

1. `district_year_diversification_features.csv`
2. `district_year_crop_share_matrix.csv`
3. `district_year_season_share_matrix.csv`

The raw and cleaned source files will not be modified.

## 1. Import libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 2. Locate and load the cleaned dataset

Expected path: `data/processed/crop_production_cleaned.csv`.

In [2]:
CLEANED_RELATIVE_PATH = (
    Path("data") / "processed" / "crop_production_cleaned.csv"
)

PROJECT_ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / CLEANED_RELATIVE_PATH).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed/crop_production_cleaned.csv. "
        "Run Notebook 02 first and check the processed folder name."
    )

CLEANED_PATH = PROJECT_ROOT / CLEANED_RELATIVE_PATH
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(CLEANED_PATH)

required_columns = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production"
]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Loaded: {CLEANED_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

display(df.head())

Loaded: c:\Users\ajadi\OneDrive\Desktop\CSD\Exploration\AgriShift\data\processed\crop_production_cleaned.csv
Rows: 246,091
Columns: 7


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,"1,254.0000","2,000.0000"
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0000,1.0000
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0000,321.0000
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0000,641.0000
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0000,165.0000


## 3. Review geographic and year-wise coverage

The source does not contain the same number of states and districts in every year. We report this explicitly.

For selected historical comparisons, we identify exact `(State_Name, District_Name)` pairs present in every year from 1997 through 2014. This is a **consistent name-based panel**, not proof that administrative boundaries remained unchanged. The dataset has no stable district identifier or boundary history.

In [3]:
DISTRICT_KEYS = ["State_Name", "District_Name"]
DISTRICT_YEAR_KEYS = ["State_Name", "District_Name", "Crop_Year"]

year_coverage = (
    df.groupby("Crop_Year")
    .agg(
        records=("Crop_Year", "size"),
        states_present=("State_Name", "nunique"),
        districts_present=("District_Name", "nunique"),
    )
    .sort_index()
)

display(year_coverage)

comparison_years = sorted(
    year for year in df["Crop_Year"].dropna().unique()
    if 1997 <= year <= 2014
)

district_sets_by_year = []
for year in comparison_years:
    year_data = df.loc[df["Crop_Year"].eq(year), DISTRICT_KEYS]
    district_sets_by_year.append(
        set(map(tuple, year_data.drop_duplicates().to_numpy()))
    )

if not district_sets_by_year:
    raise ValueError("No records found in the expected 1997–2014 comparison period.")

common_districts = set.intersection(*district_sets_by_year)

print(f"Years assessed for the consistent name-based panel: {len(comparison_years)}")
print(f"State-district pairs present in every year 1997–2014: {len(common_districts):,}")
print(f"Expected district-year observations in this panel: {len(common_districts) * len(comparison_years):,}")

,records,states_present,districts_present
Crop_Year,,,
1997,8899,23,473
1998,11533,27,495
1999,12515,28,514
2000,13658,31,543
2001,13361,31,542
2002,16671,32,565
2003,17287,32,580
2004,14117,33,585
2005,13799,33,590


Years assessed for the consistent name-based panel: 18
State-district pairs present in every year 1997–2014: 297
Expected district-year observations in this panel: 5,346


## 4. Understand why we aggregate across seasons

One crop can appear in more than one season for the same district and year. These are separate seasonal records, not necessarily duplicate rows.

For an annual crop portfolio, we add the recorded area for the same crop across its seasons. We then calculate that crop's share of the district-year total reported crop area.

In [4]:
seasons_per_crop = (
    df.groupby(DISTRICT_YEAR_KEYS + ["Crop"])["Season"]
    .nunique()
)

print(
    "District-year-crop combinations appearing in multiple seasons:",
    f"{int(seasons_per_crop.gt(1).sum()):,}"
)
print(
    "Maximum number of seasons recorded for one district-year-crop combination:",
    int(seasons_per_crop.max())
)

display(
    seasons_per_crop.value_counts()
    .sort_index()
    .rename_axis("number_of_seasons")
    .to_frame("district_year_crop_combinations")
)

District-year-crop combinations appearing in multiple seasons: 25,158
Maximum number of seasons recorded for one district-year-crop combination: 4


,district_year_crop_combinations
number_of_seasons,
1,189048
2,18449
3,6691
4,18


## 5. Aggregate area by district, year and crop

We group by state, district, year and crop. The `Area` values across all recorded seasons are added together.

The result has one row per district-year-crop combination.

In [5]:
crop_area = (
    df.groupby(DISTRICT_YEAR_KEYS + ["Crop"], as_index=False)
    .agg(
        crop_area_ha=("Area", "sum"),
        seasons_recorded=("Season", "nunique"),
        source_rows=("Crop", "size"),
    )
)

print(f"District-year-crop records after aggregation: {len(crop_area):,}")
print(f"Distinct crops retained: {crop_area['Crop'].nunique()}")
display(crop_area.head(10))

District-year-crop records after aggregation: 214,206
Distinct crops retained: 124


,State_Name,District_Name,Crop_Year,Crop,crop_area_ha,seasons_recorded,source_rows
0,Andaman and Nicobar Islands,NICOBARS,2000,Arecanut,"1,254.0000",1,1
1,Andaman and Nicobar Islands,NICOBARS,2000,Banana,176.0000,1,1
2,Andaman and Nicobar Islands,NICOBARS,2000,Cashewnut,720.0000,1,1
3,Andaman and Nicobar Islands,NICOBARS,2000,Coconut,"18,168.0000",1,1
4,Andaman and Nicobar Islands,NICOBARS,2000,Dry ginger,36.0000,1,1
5,Andaman and Nicobar Islands,NICOBARS,2000,Other Kharif pulses,2.0000,1,1
6,Andaman and Nicobar Islands,NICOBARS,2000,Rice,102.0000,1,1
7,Andaman and Nicobar Islands,NICOBARS,2000,Sugarcane,1.0000,1,1
8,Andaman and Nicobar Islands,NICOBARS,2000,Sweet potato,5.0000,1,1
9,Andaman and Nicobar Islands,NICOBARS,2000,Tapioca,40.0000,1,1


## 6. Calculate each crop's share of district-year crop area

For each district-year:

`crop_area_share = crop_area_ha / total_reported_crop_area_ha`

Example: if a district records 60,000 hectares of rice area out of 100,000 hectares of total reported crop area, rice represents 60% of that district-year's reported crop-area portfolio.

These shares should sum to 1 (or 100%) within each district-year.

In [6]:
crop_area["total_reported_crop_area_ha"] = (
    crop_area.groupby(DISTRICT_YEAR_KEYS)["crop_area_ha"].transform("sum")
)

if (crop_area["total_reported_crop_area_ha"] <= 0).any():
    raise ValueError("A district-year has zero or negative total reported crop area.")

crop_area["crop_area_share"] = (
    crop_area["crop_area_ha"] / crop_area["total_reported_crop_area_ha"]
)

share_sum_check = (
    crop_area.groupby(DISTRICT_YEAR_KEYS)["crop_area_share"]
    .sum()
    .rename("sum_of_crop_shares")
)

print("Minimum share sum:", share_sum_check.min())
print("Maximum share sum:", share_sum_check.max())
print("All district-year crop shares sum to 1:",
      bool(np.allclose(share_sum_check.to_numpy(), 1.0)))

display(crop_area.head(10))

Minimum share sum: 0.9999999999999997
Maximum share sum: 1.0000000000000004
All district-year crop shares sum to 1: True


,State_Name,District_Name,Crop_Year,Crop,crop_area_ha,seasons_recorded,source_rows,total_reported_crop_area_ha,crop_area_share
0,Andaman and Nicobar Islands,NICOBARS,2000,Arecanut,"1,254.0000",1,1,"20,504.0000",0.0612
1,Andaman and Nicobar Islands,NICOBARS,2000,Banana,176.0000,1,1,"20,504.0000",0.0086
2,Andaman and Nicobar Islands,NICOBARS,2000,Cashewnut,720.0000,1,1,"20,504.0000",0.0351
3,Andaman and Nicobar Islands,NICOBARS,2000,Coconut,"18,168.0000",1,1,"20,504.0000",0.8861
4,Andaman and Nicobar Islands,NICOBARS,2000,Dry ginger,36.0000,1,1,"20,504.0000",0.0018
5,Andaman and Nicobar Islands,NICOBARS,2000,Other Kharif pulses,2.0000,1,1,"20,504.0000",0.0001
6,Andaman and Nicobar Islands,NICOBARS,2000,Rice,102.0000,1,1,"20,504.0000",0.0050
7,Andaman and Nicobar Islands,NICOBARS,2000,Sugarcane,1.0000,1,1,"20,504.0000",0.0000
8,Andaman and Nicobar Islands,NICOBARS,2000,Sweet potato,5.0000,1,1,"20,504.0000",0.0002
9,Andaman and Nicobar Islands,NICOBARS,2000,Tapioca,40.0000,1,1,"20,504.0000",0.0020


## 7. Calculate crop diversification and concentration indicators

- **Crop richness:** number of distinct crops recorded.
- **Shannon entropy:** reflects both the number of crops and how evenly area is distributed.
- **Normalized Shannon evenness:** Shannon entropy divided by the maximum possible entropy for the observed crop count. It ranges from 0 to 1.
- **HHI concentration:** sum of squared crop shares. Higher values mean greater concentration.
- **Simpson diversity:** 1 minus HHI. Higher values mean less concentration.
- **Effective number of crops:** exponential of Shannon entropy. It expresses diversity as the equivalent number of equally represented crops.
- **Top-one and top-three shares:** proportion of reported crop area represented by the largest one or three crops.
- **Dominant crop:** crop with the largest reported area share.

No single index tells the whole story, so we retain several indicators.

In [7]:
feature_rows = []

for group_key, group in crop_area.groupby(DISTRICT_YEAR_KEYS, sort=False):
    state, district, year = group_key
    ranked = group.sort_values("crop_area_share", ascending=False)
    shares = ranked["crop_area_share"].to_numpy(dtype=float)
    shares = shares[shares > 0]

    crop_count = len(shares)
    shannon_entropy = float(-(shares * np.log(shares)).sum())
    hhi = float(np.square(shares).sum())
    normalized_evenness = (
        shannon_entropy / np.log(crop_count)
        if crop_count > 1 else 0.0
    )

    feature_rows.append({
        "State_Name": state,
        "District_Name": district,
        "Crop_Year": int(year),
        "total_reported_crop_area_ha": float(ranked["total_reported_crop_area_ha"].iloc[0]),
        "crop_richness": int(crop_count),
        "shannon_entropy": shannon_entropy,
        "normalized_shannon_evenness": float(normalized_evenness),
        "hhi_concentration": hhi,
        "simpson_diversity": 1.0 - hhi,
        "effective_number_of_crops": float(np.exp(shannon_entropy)),
        "top_1_crop_share": float(shares[0]),
        "top_3_crop_share": float(shares[:3].sum()),
        "dominant_crop": str(ranked["Crop"].iloc[0]),
        "crop_count_records": int(len(group)),
    })

crop_diversity_features = pd.DataFrame(feature_rows)

print("District-year diversification feature table:")
print(crop_diversity_features.shape)
display(crop_diversity_features.head())

District-year diversification feature table:
(9839, 14)


,State_Name,District_Name,Crop_Year,total_reported_crop_area_ha,crop_richness,shannon_entropy,normalized_shannon_evenness,hhi_concentration,simpson_diversity,effective_number_of_crops,top_1_crop_share,top_3_crop_share,dominant_crop,crop_count_records
0,Andaman and Nicobar Islands,NICOBARS,2000,"20,504.0000",10,0.4896,0.2126,0.7902,0.2098,1.6317,0.8861,0.9823,Coconut,10
1,Andaman and Nicobar Islands,NICOBARS,2001,"20,306.0000",8,0.4306,0.2071,0.8075,0.1925,1.5381,0.8958,0.9930,Coconut,8
2,Andaman and Nicobar Islands,NICOBARS,2002,"21,147.5000",9,0.6087,0.2770,0.7492,0.2508,1.8381,0.8625,0.9560,Coconut,9
3,Andaman and Nicobar Islands,NICOBARS,2003,"20,822.4400",9,0.5356,0.2438,0.7762,0.2238,1.7085,0.8781,0.9731,Coconut,9
4,Andaman and Nicobar Islands,NICOBARS,2004,"20,726.2600",9,0.4921,0.2240,0.7926,0.2074,1.6357,0.8875,0.9815,Coconut,9


## 8. Calculate seasonal distribution features

Crop composition and season composition are related but different questions.

We aggregate reported area by season for each district-year and calculate each season's share. We also calculate seasonal richness, entropy and concentration.

These are separate descriptive features so we can study seasonal dependence without confusing it with crop diversity.

In [8]:
season_area = (
    df.groupby(DISTRICT_YEAR_KEYS + ["Season"], as_index=False)
    .agg(season_area_ha=("Area", "sum"))
)

season_area["total_reported_crop_area_ha"] = (
    season_area.groupby(DISTRICT_YEAR_KEYS)["season_area_ha"].transform("sum")
)

season_area["season_area_share"] = (
    season_area["season_area_ha"] / season_area["total_reported_crop_area_ha"]
)

season_feature_rows = []

for group_key, group in season_area.groupby(DISTRICT_YEAR_KEYS, sort=False):
    state, district, year = group_key
    ranked = group.sort_values("season_area_share", ascending=False)
    shares = ranked["season_area_share"].to_numpy(dtype=float)
    shares = shares[shares > 0]

    season_count = len(shares)
    entropy = float(-(shares * np.log(shares)).sum())
    hhi = float(np.square(shares).sum())
    evenness = entropy / np.log(season_count) if season_count > 1 else 0.0

    season_feature_rows.append({
        "State_Name": state,
        "District_Name": district,
        "Crop_Year": int(year),
        "season_richness": int(season_count),
        "season_shannon_entropy": entropy,
        "normalized_season_evenness": float(evenness),
        "season_hhi_concentration": hhi,
        "dominant_season": str(ranked["Season"].iloc[0]),
    })

season_features = pd.DataFrame(season_feature_rows)

season_share_matrix = (
    season_area.pivot_table(
        index=DISTRICT_YEAR_KEYS,
        columns="Season",
        values="season_area_share",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)
season_share_matrix.columns.name = None

print("Season feature table:")
display(season_features.head())
print("Season share matrix shape:", season_share_matrix.shape)
display(season_share_matrix.head())

Season feature table:


,State_Name,District_Name,Crop_Year,season_richness,season_shannon_entropy,normalized_season_evenness,season_hhi_concentration,dominant_season
0,Andaman and Nicobar Islands,NICOBARS,2000,2,0.2438,0.3517,0.8763,Whole Year
1,Andaman and Nicobar Islands,NICOBARS,2001,2,0.2430,0.3506,0.8768,Whole Year
2,Andaman and Nicobar Islands,NICOBARS,2002,2,0.0511,0.0737,0.9823,Whole Year
3,Andaman and Nicobar Islands,NICOBARS,2003,2,0.0175,0.0252,0.9950,Whole Year
4,Andaman and Nicobar Islands,NICOBARS,2004,2,0.0178,0.0257,0.9949,Whole Year


Season share matrix shape: (9839, 9)


,State_Name,District_Name,Crop_Year,Autumn,Kharif,Rabi,Summer,Whole Year,Winter
0,Andaman and Nicobar Islands,NICOBARS,2000,0.0000,0.0662,0.0000,0.0000,0.9338,0.0000
1,Andaman and Nicobar Islands,NICOBARS,2001,0.0000,0.0659,0.0000,0.0000,0.9341,0.0000
2,Andaman and Nicobar Islands,NICOBARS,2002,0.0000,0.0089,0.0000,0.0000,0.9911,0.0000
3,Andaman and Nicobar Islands,NICOBARS,2003,0.0000,0.0025,0.0000,0.0000,0.9975,0.0000
4,Andaman and Nicobar Islands,NICOBARS,2004,0.0000,0.0026,0.0000,0.0000,0.9974,0.0000


## 9. Create the district-year crop-share matrix

Clustering algorithms need one row per observation and one column per feature.

We pivot the crop shares so that:
- Each row represents one state-district-year.
- Each crop becomes a column.
- Each cell contains that crop's share of reported crop area.
- A zero means no area for that crop was recorded in that district-year.

This creates the crop-composition feature matrix for later clustering.

In [9]:
crop_share_matrix = (
    crop_area.pivot_table(
        index=DISTRICT_YEAR_KEYS,
        columns="Crop",
        values="crop_area_share",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

crop_share_matrix.columns.name = None
crop_columns = [
    column for column in crop_share_matrix.columns
    if column not in DISTRICT_YEAR_KEYS
]

row_share_sums = crop_share_matrix[crop_columns].sum(axis=1)

print("Crop-share matrix shape:", crop_share_matrix.shape)
print("Number of crop feature columns:", len(crop_columns))
print("Minimum row share sum:", row_share_sums.min())
print("Maximum row share sum:", row_share_sums.max())
print("All rows sum to 1:",
      bool(np.allclose(row_share_sums.to_numpy(), 1.0)))

display(crop_share_matrix.head())

Crop-share matrix shape: (9839, 127)
Number of crop feature columns: 124
Minimum row share sum: 0.9999999999999998
Maximum row share sum: 1.0000000000000002
All rows sum to 1: True


,State_Name,District_Name,Crop_Year,Apple,Arcanut (Processed),Arecanut,Arhar/Tur,Ash Gourd,Atcanut (Raw),Bajra,Banana,Barley,Bean,Beans & Mutter(Vegetable),Beet Root,Ber,Bhindi,Bitter Gourd,Black pepper,Blackgram,Bottle Gourd,Brinjal,Cabbage,Cardamom,Carrot,Cashewnut,Cashewnut Processed,Cashewnut Raw,Castor seed,Cauliflower,Citrus Fruit,Coconut,Coffee,Colocosia,Cond-spcs other,Coriander,Cotton(lint),Cowpea(Lobia),Cucumber,Drum Stick,Dry chillies,Dry ginger,Garlic,Ginger,Gram,Grapes,Groundnut,Guar seed,Horse-gram,Jack Fruit,Jobster,Jowar,Jute,Jute & mesta,Kapas,Khesari,Korra,Lab-Lab,Lemon,Lentil,Linseed,Litchi,Maize,Mango,Masoor,Mesta,Moong(Green Gram),Moth,Niger seed,Oilseeds total,Onion,Orange,Other Cereals & Millets,Other Citrus Fruit,Other Dry Fruit,Other Fresh Fruits,Other Kharif pulses,Other Rabi pulses,Other Vegetables,Paddy,Papaya,Peach,Pear,Peas & beans (Pulses),Peas (vegetable),Perilla,Pineapple,Plums,Pome Fruit,Pome Granet,Potato,Pulses total,Pump Kin,Ragi,Rajmash Kholar,Rapeseed &Mustard,Redish,Ribed Guard,Rice,Ricebean (nagadal),Rubber,Safflower,Samai,Sannhamp,Sapota,Sesamum,Small millets,Snak Guard,Soyabean,Sugarcane,Sunflower,Sweet potato,Tapioca,Tea,Tobacco,Tomato,Total foodgrain,Turmeric,Turnip,Urad,Varagu,Water Melon,Wheat,Yam,other fibres,other misc. pulses,other oilseeds
0,Andaman and Nicobar Islands,NICOBARS,2000,0.0000,0.0000,0.0612,0.0000,0.0000,0.0000,0.0000,0.0086,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0351,0.0000,0.0000,0.0000,0.0000,0.0000,0.8861,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0018,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0050,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0002,0.0020,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,Andaman and Nicobar Islands,NICOBARS,2001,0.0000,0.0000,0.0618,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0354,0.0000,0.0000,0.0000,0.0000,0.0000,0.8958,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0023,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0041,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,Andaman and Nicobar Islands,NICOBARS,2002,0.0000,0.0000,0.0595,0.0000,0.0000,0.0000,0.0000,0.0101,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0030,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0340,0.0000,0.0000,0.0000,0.0000,0.0000,0.8625,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0195,0.0022,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0089,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.

## 10. Combine the district-year feature tables

In [10]:
diversification_features = crop_diversity_features.merge(
    season_features,
    on=DISTRICT_YEAR_KEYS,
    how="left",
    validate="one_to_one",
)

diversification_features["in_common_name_panel_1997_2014"] = (
    diversification_features.apply(
        lambda row: (
            (row["State_Name"], row["District_Name"]) in common_districts
            and 1997 <= row["Crop_Year"] <= 2014
        ),
        axis=1,
    )
)

print("Combined feature table shape:", diversification_features.shape)
print(
    "District-year observations in common name-based panel:",
    int(diversification_features["in_common_name_panel_1997_2014"].sum())
)
display(diversification_features.head())

Combined feature table shape: (9839, 20)
District-year observations in common name-based panel: 5346


,State_Name,District_Name,Crop_Year,total_reported_crop_area_ha,crop_richness,shannon_entropy,normalized_shannon_evenness,hhi_concentration,simpson_diversity,effective_number_of_crops,top_1_crop_share,top_3_crop_share,dominant_crop,crop_count_records,season_richness,season_shannon_entropy,normalized_season_evenness,season_hhi_concentration,dominant_season,in_common_name_panel_1997_2014
0,Andaman and Nicobar Islands,NICOBARS,2000,"20,504.0000",10,0.4896,0.2126,0.7902,0.2098,1.6317,0.8861,0.9823,Coconut,10,2,0.2438,0.3517,0.8763,Whole Year,False
1,Andaman and Nicobar Islands,NICOBARS,2001,"20,306.0000",8,0.4306,0.2071,0.8075,0.1925,1.5381,0.8958,0.9930,Coconut,8,2,0.2430,0.3506,0.8768,Whole Year,False
2,Andaman and Nicobar Islands,NICOBARS,2002,"21,147.5000",9,0.6087,0.2770,0.7492,0.2508,1.8381,0.8625,0.9560,Coconut,9,2,0.0511,0.0737,0.9823,Whole Year,False
3,Andaman and Nicobar Islands,NICOBARS,2003,"20,822.4400",9,0.5356,0.2438,0.7762,0.2238,1.7085,0.8781,0.9731,Coconut,9,2,0.0175,0.0252,0.9950,Whole Year,False
4,Andaman and Nicobar Islands,NICOBARS,2004,"20,726.2600",9,0.4921,0.2240,0.7926,0.2074,1.6357,0.8875,0.9815,Coconut,9,2,0.0178,0.0257,0.9949,Whole Year,False


## 11. Validate the engineered features

In [11]:
expected_district_years = df[DISTRICT_YEAR_KEYS].drop_duplicates().shape[0]

assert len(diversification_features) == expected_district_years
assert len(crop_share_matrix) == expected_district_years
assert np.allclose(row_share_sums.to_numpy(), 1.0)
assert crop_share_matrix[crop_columns].ge(0).all().all()
assert crop_share_matrix[crop_columns].le(1).all().all()

assert diversification_features["normalized_shannon_evenness"].between(0, 1).all()
assert diversification_features["hhi_concentration"].between(0, 1).all()
assert diversification_features["simpson_diversity"].between(0, 1).all()
assert diversification_features["normalized_season_evenness"].between(0, 1).all()

print("Feature validation passed.")
print(f"District-year observations: {expected_district_years:,}")
print(f"Crop share features: {len(crop_columns)}")
print(
    "Common name-based panel observations:",
    int(diversification_features["in_common_name_panel_1997_2014"].sum())
)

Feature validation passed.
District-year observations: 9,839
Crop share features: 124
Common name-based panel observations: 5346


## 12. Inspect the engineered indicators

In [12]:
display(
    diversification_features[
        [
            "State_Name", "District_Name", "Crop_Year",
            "crop_richness", "normalized_shannon_evenness",
            "hhi_concentration", "effective_number_of_crops",
            "top_1_crop_share", "top_3_crop_share",
            "dominant_crop", "dominant_season",
            "in_common_name_panel_1997_2014",
        ]
    ].head(15)
)

print("\nMost concentrated district-year portfolios (highest HHI):")
display(
    diversification_features
    .sort_values("hhi_concentration", ascending=False)
    [[
        "State_Name", "District_Name", "Crop_Year",
        "crop_richness", "hhi_concentration",
        "top_1_crop_share", "dominant_crop"
    ]]
    .head(10)
)

,State_Name,District_Name,Crop_Year,crop_richness,normalized_shannon_evenness,hhi_concentration,effective_number_of_crops,top_1_crop_share,top_3_crop_share,dominant_crop,dominant_season,in_common_name_panel_1997_2014
0,Andaman and Nicobar Islands,NICOBARS,2000,10,0.2126,0.7902,1.6317,0.8861,0.9823,Coconut,Whole Year,False
1,Andaman and Nicobar Islands,NICOBARS,2001,8,0.2071,0.8075,1.5381,0.8958,0.9930,Coconut,Whole Year,False
2,Andaman and Nicobar Islands,NICOBARS,2002,9,0.2770,0.7492,1.8381,0.8625,0.9560,Coconut,Whole Year,False
3,Andaman and Nicobar Islands,NICOBARS,2003,9,0.2438,0.7762,1.7085,0.8781,0.9731,Coconut,Whole Year,False
4,Andaman and Nicobar Islands,NICOBARS,2004,9,0.2240,0.7926,1.6357,0.8875,0.9815,Coconut,Whole Year,False
5,Andaman and Nicobar Islands,NICOBARS,2005,8,0.1834,0.8355,1.4644,0.9121,0.9939,Coconut,Whole Year,False
6,Andaman and Nicobar Islands,NICOBARS,2006,11,0.1898,0.8088,1.5762,0.8970,0.9811,Coconut,Whole Year,False
7,Andaman and Nicobar Islands,NICOBARS,2010,15,0.2373,0.7195,1.9016,0.8438,0.9564,Coconut,Whole Year,False
8,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,2000,11,0.6047,0.3081,4.2635,0.4489,0.8691,Rice,Kharif,False
9,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,2001,8,0.5869,0.3530,3.3887,0.4659,0.9509,Rice,Kharif,False



Most concentrated district-year portfolios (highest HHI):


,State_Name,District_Name,Crop_Year,crop_richness,hhi_concentration,top_1_crop_share,dominant_crop
5375,Manipur,IMPHAL WEST,1997,1,1.0000,1.0000,Rice
5026,Maharashtra,MUMBAI,1997,1,1.0000,1.0000,Maize
5115,Maharashtra,PALGHAR,2011,1,1.0000,1.0000,Groundnut
3646,Karnataka,UDUPI,1999,1,1.0000,1.0000,Rice
3944,Madhya Pradesh,ANUPPUR,2002,1,1.0000,1.0000,Sugarcane
7143,Rajasthan,PRATAPGARH,1999,1,1.0000,1.0000,Banana
7144,Rajasthan,PRATAPGARH,2001,1,1.0000,1.0000,Banana
6703,Punjab,PATHANKOT,2011,1,1.0000,1.0000,Sugarcane
3008,Jammu and Kashmir,SHOPIAN,2014,1,1.0000,1.0000,Potato
6430,Puducherry,YANAM,2006,1,1.0000,1.0000,Rice


## 13. Save the engineered datasets

In [13]:
OUTPUT_FILES = {
    "diversification_features": PROCESSED_DIR / "district_year_diversification_features.csv",
    "crop_share_matrix": PROCESSED_DIR / "district_year_crop_share_matrix.csv",
    "season_share_matrix": PROCESSED_DIR / "district_year_season_share_matrix.csv",
}

diversification_features.to_csv(
    OUTPUT_FILES["diversification_features"], index=False
)
crop_share_matrix.to_csv(
    OUTPUT_FILES["crop_share_matrix"], index=False
)
season_share_matrix.to_csv(
    OUTPUT_FILES["season_share_matrix"], index=False
)

for label, path in OUTPUT_FILES.items():
    print(f"{label}: {path}")
    print(f"  Size: {path.stat().st_size / (1024 * 1024):.2f} MB")

diversification_features: c:\Users\ajadi\OneDrive\Desktop\CSD\Exploration\AgriShift\data\processed\district_year_diversification_features.csv
  Size: 2.35 MB
crop_share_matrix: c:\Users\ajadi\OneDrive\Desktop\CSD\Exploration\AgriShift\data\processed\district_year_crop_share_matrix.csv
  Size: 8.42 MB
season_share_matrix: c:\Users\ajadi\OneDrive\Desktop\CSD\Exploration\AgriShift\data\processed\district_year_season_share_matrix.csv
  Size: 1.00 MB


## 14. Reopen and verify the saved outputs

In [14]:
features_check = pd.read_csv(OUTPUT_FILES["diversification_features"])
crop_matrix_check = pd.read_csv(OUTPUT_FILES["crop_share_matrix"])
season_matrix_check = pd.read_csv(OUTPUT_FILES["season_share_matrix"])

assert features_check.shape == diversification_features.shape
assert crop_matrix_check.shape == crop_share_matrix.shape
assert season_matrix_check.shape == season_share_matrix.shape

print("All three feature files were saved and read back successfully.")
print("Diversification features:", features_check.shape)
print("Crop-share matrix:", crop_matrix_check.shape)
print("Season-share matrix:", season_matrix_check.shape)

All three feature files were saved and read back successfully.
Diversification features: (9839, 20)
Crop-share matrix: (9839, 127)
Season-share matrix: (9839, 9)


## 15. Interpretation and limitations

### How to read the indicators
- Higher `crop_richness` means more crop types are recorded.
- Higher `normalized_shannon_evenness` means reported crop area is distributed more evenly across the crops present.
- Higher `hhi_concentration` means a larger share of reported crop area is concentrated in fewer crops.
- Higher `simpson_diversity` means less concentration.
- `effective_number_of_crops` translates Shannon entropy into the equivalent number of equally represented crops.
- `top_1_crop_share` and `top_3_crop_share` show the share of reported crop area belonging to the largest one or three crops.

### Important limitations
1. These are descriptive indicators, not causal explanations.
2. Shares are based on reported crop area summed across seasons, not unique physical district land.
3. The number of states and districts varies by year.
4. The common 1997–2014 panel matches exact state and district names; it cannot guarantee stable administrative boundaries.
5. Crop labels have not been manually merged into broader crop families.

### Stage outcome
We have transformed the cleaned source records into district-year crop portfolios, diversification indicators and crop/season share matrices. These outputs will be used for exploratory analysis and later clustering.